# 04 · Vector Store: FAISS

**FAISS** (Facebook AI Similarity Search) is a library for fast similarity search over dense vectors.
It runs **locally and in memory**, and the index can be saved to disk and loaded again.

> RAG pipeline: Load ➜ Split ➜ Embed ➜ **Store ➜ Retrieve**

### What this notebook covers
1. Load and split `speech.txt`
2. Embed the chunks with Ollama (`nomic-embed-text`) and build a FAISS index
3. `similarity_search` and `similarity_search_with_score`
4. Turn the vector store into a **Retriever**
5. Save the index to disk and load it back

**Requirements:** `faiss-cpu`, plus Ollama running with `nomic-embed-text` pulled.

## 1. Imports and loading the document

> ⚠️ `from langchain_community.embeddings import OllamaEmbeddings` is deprecated (this causes the warning in cell 3).
> The recommended import is `from langchain_ollama import OllamaEmbeddings`.

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
documents = loader.load()

## 2. Split with `CharacterTextSplitter`

Unlike the recursive splitter, `CharacterTextSplitter` splits on **one separator only** (default `"\n\n"`) and then merges pieces up to `chunk_size`.
If a single paragraph is longer than `chunk_size`, it stays as one oversized chunk (with a warning).

In [7]:
text_splitter = CharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 30
)

docs =  text_splitter.split_documents(documents)

In [8]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'),
 Document(metadata={'source': 'speech.txt'}, page_content='One of the most important concepts in AI is Machine Learning. Machine Learning is a technique that allows computers to learn from data without being explicitly programmed for every task. For example, email spam filters use machine learning to identify unwanted emails based on previously observed patterns.'),
 Document(metadata={'source': 'sp

## 3. Embedding model
Local Ollama model `nomic-embed-text` (768 dimensions).

In [9]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

C:\Users\yokes\AppData\Local\Temp\ipykernel_28776\2497687198.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(


## 4. Create the FAISS index
`FAISS.from_documents()` embeds every chunk and builds the index. By default it uses a flat L2 index, which is exact (brute-force) search.

In [10]:
db = FAISS.from_documents(
    docs,
    embeddings
)

## 5. Similarity search
Embeds the query and returns the `k=4` (default) most similar chunks. `docs[0]` is the best match.

In [16]:
query = "What is Artificial intelligence"

docs = db.similarity_search(
    query
)
docs[0]

Document(id='cff64ae1-3396-43b7-bfd7-c20ae96a137d', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.')

## 6. Retriever

`as_retriever()` wraps the vector store in LangChain's standard **Retriever** interface (a *Runnable*), so it can be plugged into chains and called with `.invoke(query)`.

Common options:
```python
db.as_retriever(search_type="mmr", search_kwargs={"k": 2})
```

In [17]:
### Retrievers

retriever = db.as_retriever()
retriever.invoke(query)

[Document(id='cff64ae1-3396-43b7-bfd7-c20ae96a137d', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'),
 Document(id='bedae169-aa55-4c7e-a499-05cd2063eba5', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.'),
 Document(id='8624d873-09ed-4aa9-b444-45e356cfb20a', metadata={'source': 'speech.txt'}, page_content='However, Artificial Intelligence also comes with challenges. These include data privacy, security, bias, misinformation, and the responsible use of AI technologies.'),
 Document(id='4f0dac8c-5d3b-48ef-96b8-8879d79f

## 7. Similarity search with score

Returns `(Document, score)` tuples. With FAISS, the default score is **L2 distance**, so **a lower score means more similar**.

In [18]:
## Similarity Search with score 
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='cff64ae1-3396-43b7-bfd7-c20ae96a137d', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'),
  np.float32(310.27844)),
 (Document(id='bedae169-aa55-4c7e-a499-05cd2063eba5', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.'),
  np.float32(323.18927)),
 (Document(id='8624d873-09ed-4aa9-b444-45e356cfb20a', metadata={'source': 'speech.txt'}, page_content='However, Artificial Intelligence also comes with challenges. These include data privacy, security, bias, misinformation, and the responsible use of AI technolo

## 8. Save and load the index

`save_local("faiss_index")` creates a folder with:
- `index.faiss`: the vectors
- `index.pkl`: the documents and id mapping (Python pickle)

When loading, pass the **same embedding model** that built the index.
`allow_dangerous_deserialization=True` is required because pickle files can execute code. **Only load index files you created yourself.**

> `faiss_index/` is in `.gitignore`. Running this notebook recreates it.

In [19]:
## saving and loading
db.save_local("faiss_index")

In [22]:
new_db = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True)

Query the reloaded index. It gives the same results as the original.

In [23]:
docs = new_db.similarity_search(query)

In [24]:
docs

[Document(id='cff64ae1-3396-43b7-bfd7-c20ae96a137d', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence is a branch of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, understanding language, and recognizing images.'),
 Document(id='bedae169-aa55-4c7e-a499-05cd2063eba5', metadata={'source': 'speech.txt'}, page_content='Artificial Intelligence: Concepts and Applications\n\nGood morning everyone!\n\nToday, I am going to talk about Artificial Intelligence, commonly known as AI, and some of its important concepts.'),
 Document(id='8624d873-09ed-4aa9-b444-45e356cfb20a', metadata={'source': 'speech.txt'}, page_content='However, Artificial Intelligence also comes with challenges. These include data privacy, security, bias, misinformation, and the responsible use of AI technologies.'),
 Document(id='4f0dac8c-5d3b-48ef-96b8-8879d79f

## Summary

| Method | Returns |
|---|---|
| `FAISS.from_documents(docs, embeddings)` | Builds the vector store |
| `db.similarity_search(query, k=4)` | `list[Document]` |
| `db.similarity_search_with_score(query)` | `list[(Document, L2 distance)]` (lower = better) |
| `db.as_retriever()` | Retriever for use in chains (`.invoke`) |
| `db.save_local(path)` / `FAISS.load_local(path, embeddings, ...)` | Persist / reload |

### FAISS vs Chroma (both used in this folder)
| | FAISS | Chroma |
|---|---|---|
| Type | Library (index in memory) | Lightweight vector database |
| Persistence | `save_local` / `load_local` | `persist_directory` |
| Metadata filtering | Basic | Built-in |

➡️ **Next step:** connect the retriever to an LLM with a prompt to build a complete **RAG chain**.